##**1. Data Collection**

In [ ]:
pip install pandas numpy

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import re

In [ ]:
# Utility Function to Load and Rename CSVs
def load_and_rename(filepath, columns, rename_map, encoding='ISO-8859-1', skiprows=None):
    df = pd.read_csv(filepath, encoding=encoding, skiprows=skiprows)
    df = df.loc[:, columns].rename(columns=rename_map)
    return df

###**1.1 Demographics**

In [ ]:
# Load and process LSOA data
lsoa_df = load_and_rename(
    'lsoa-data.csv',
    columns=['Lower Super Output Area', 'Names', 'Economic Activity;Employment Rate;2011', 'Public Transport Accessibility Levels (2014);Average Score;'],
    rename_map={'Lower Super Output Area': 'LSOA Code', 'Names': 'LSOA Name', 'Economic Activity;Employment Rate;2011': 'Employment Rate 2011', 'Public Transport Accessibility Levels (2014);Average Score;': 'PT Accessibility Levels 2014'}
)

# Load and process Population 2015 data
population_df = load_and_rename(
    'Population 2015 (ONS).csv',
    columns=['LSOA code (2011)', 'Total population: mid 2015 (excluding prisoners)', 'Dependent Children aged 0-15: mid 2015 (excluding prisoners)', 'Population aged 16-59: mid 2015 (excluding prisoners)', 'Older population aged 60 and over: mid 2015 (excluding prisoners)'],
    rename_map={'LSOA code (2011)': 'LSOA Code', 'Total population: mid 2015 (excluding prisoners)': 'Population 2015', 'Dependent Children aged 0-15: mid 2015 (excluding prisoners)': 'Children 0-15', 'Population aged 16-59: mid 2015 (excluding prisoners)': 'Population 16-59', 'Older population aged 60 and over: mid 2015 (excluding prisoners)': 'Population 60+'}
)

# Merge LSOA and Population data, then calculate Average Age for 2015
lsoa_df = pd.merge(lsoa_df, population_df, on='LSOA Code')
lsoa_df['Average Age 2015'] = (lsoa_df['Children 0-15'] * 7.5 + lsoa_df['Population 16-59'] * 37.5 + lsoa_df['Population 60+'] * 70) / lsoa_df['Population 2015']
lsoa_df = lsoa_df.drop(columns=['Children 0-15', 'Population 16-59', 'Population 60+'])

# Load and merge Median House Price 2023
median_price_df = load_and_rename('Median price.csv', ['LSOA code', 'Year ending Mar 2023'], {'LSOA code': 'LSOA Code', 'Year ending Mar 2023': 'Median House Price 2023'}, skiprows=5)

lsoa_data = pd.merge(lsoa_df, median_price_df, on='LSOA Code').dropna()

###**1.2 Mapping postcodes to LSOA**

In [ ]:
# Load and process London Postcode Data with additional columns
postcodes = pd.read_csv('London postcodes.csv', low_memory=False)[
    ['Postcode', 'LSOA Code', 'Lower layer super output area',
     'District', 'District Code', 'London zone', 'Distance to station',
     'Average Income', 'Latitude', 'Longitude', 'Index of Multiple Deprivation']
]

# Rename columns to standardize naming conventions
postcodes = postcodes.rename(columns={
    'Lower layer super output area': 'LSOA Name',
    'District': 'District',
    'District Code': 'District Code',
    'London zone': 'London Zone',
    'Distance to station': 'Distance to Station',
    'Latitude': 'Latitude',
    'Longitude': 'Longitude',
    'Index of Multiple Deprivation': 'Index of Multiple Deprivation'
})

# Group by LSOA Code to concatenate Postcodes and aggregate other columns if necessary
postcodes_grouped = postcodes.groupby('LSOA Code').agg({
    'Postcode': ', '.join,
    'LSOA Name': 'first',
    'District': 'first',
    'District Code': 'first',
    'London Zone': 'first',
    'Distance to Station': 'first',
    'Average Income': 'first',
    'Latitude': 'first',
    'Longitude': 'first',
    'Index of Multiple Deprivation': 'first'
}).reset_index()

# Merge the additional postcode data into the lsoa_data DataFrame
lsoa_data = pd.merge(lsoa_data, postcodes_grouped, on='LSOA Code', how='left')

# If there's duplication, handle the LSOA Name columns
lsoa_data = lsoa_data.drop(columns=['LSOA Name_y'])
lsoa_data = lsoa_data.rename(columns={'LSOA Name_x': 'LSOA Name'})

# Drop any rows with missing values if necessary
lsoa_data = lsoa_data.dropna()

# Save the updated dataset to a CSV file
lsoa_data.to_csv('LSOA Data Updated.csv', index=False)
print("The updated LSOA data with additional columns has been saved to 'LSOA Data Updated.csv'.")

###**1.3 Competitors Data -> Apify**


In [ ]:
# Load competitor data from multiple files
competitor_files = [
    'Google Maps Extractor (Cafe, Coffee Shop, Bakery, Tea room).csv',
    'Google Maps Scraper ((Cafe, Coffee Shop, Coffee Store, Bubble Tea Store).csv',
    'Google Maps Scraper (Cafe, Coffee Shop, Bakery, Tea room).csv',
    'Google Maps Scraper (Cafe, Coffee Shop).csv',
    'Google Maps Scraper (Cafe).csv',
    'Google Maps Scraper (Coffee Shop).csv'
]
competitors = pd.concat([pd.read_csv(file) for file in competitor_files], ignore_index=True)

# Extract postcodes and clean data
def extract_postcode(address):
    match = re.search(r'\b[A-Z]{1,2}[0-9][0-9A-Z]?\s?[0-9][A-Z]{2}\b', str(address))
    return match.group(0) if match else None

competitors['postalCode'] = competitors.apply(lambda row: row['postalCode'] if pd.notnull(row['postalCode']) else extract_postcode(row['address']), axis=1)
competitors = competitors.dropna(subset=['postalCode']).drop(columns=['location']).drop_duplicates()
competitors['postalCode'] = competitors['postalCode'].str.strip()
lsoa_data['Postcode'] = lsoa_data['Postcode'].str.strip()

# Map competitor postcodes to LSOA Codes and aggregate metrics
lsoa_postcode_mapping = {pc: row['LSOA Code'] for _, row in lsoa_data.iterrows() for pc in row['Postcode'].split(', ')}
competitors['LSOA Code'] = competitors['postalCode'].map(lsoa_postcode_mapping)
lsoa_metrics = competitors.groupby('LSOA Code').agg(Competitors=('title', 'size'), Cafe_Score=('totalScore', 'mean'), Reviews=('reviewsCount', 'sum')).reset_index()
lsoa_data = pd.merge(lsoa_data, lsoa_metrics, on='LSOA Code', how='left').fillna({'Competitors': 0, 'Cafe_Score': 0, 'Reviews': 0})


###**1.4 Amenities Data -> Apify**

In [ ]:
# Load and process hotel data
hotel_files = ['TripAdvisor Hotels in Greater London.csv', 'TripAdvisor Hotels in London.csv']
hotels = pd.concat([pd.read_csv(file) for file in hotel_files], ignore_index=True).drop_duplicates()
hotels = hotels[['location/locationV2/names/name', 'location/locationV2/contact/streetAddress/postalCode', 'location/locationV2/geocode/latitude', 'location/locationV2/geocode/longitude']]
hotels = hotels.rename(columns={'location/locationV2/names/name': 'Name', 'location/locationV2/contact/streetAddress/postalCode': 'Postal Code', 'location/locationV2/geocode/latitude': 'Latitude', 'location/locationV2/geocode/longitude': 'Longitude'})
hotels['Category'] = 'Hotel'

# Load and process station data
stations = pd.read_csv('Stations.csv')[['Name', 'Postal Code', 'Latitude', 'Longitude']]
stations['Category'] = 'Station'

# Load and process additional amenities data
amenity_files = [
    'Google Maps Scraper (Business Center, Shopping Center, Tourist Attration).csv',
    'Google Maps Scraper (University, College, Museum, Gallery, Theatre, Cinema, Gym).csv'
]
amenities = pd.concat([pd.read_csv(file) for file in amenity_files], ignore_index=True)
amenities['Postal Code'] = amenities.apply(lambda row: row['postalCode'] if pd.notnull(row['postalCode']) else extract_postcode(row['address']), axis=1)
amenities = amenities.dropna(subset=['Postal Code'])
amenities = amenities[['title', 'postalCode', 'location/lat', 'location/lng', 'searchString']]
amenities = amenities.rename(columns={'title': 'Name', 'postalCode': 'Postal Code', 'location/lat': 'Latitude', 'location/lng': 'Longitude', 'searchString': 'Category'})

# Combine relevant columns from hotels, stations, and amenities
hotels = hotels[['Name', 'Postal Code', 'Latitude', 'Longitude', 'Category']]
stations = stations[['Name', 'Postal Code', 'Latitude', 'Longitude', 'Category']]
amenities = amenities[['Name', 'Postal Code', 'Latitude', 'Longitude', 'Category']]

# Combine amenities data, reset indices, normalize postcodes, and map to LSOA Codes
hotels.reset_index(drop=True, inplace=True)
stations.reset_index(drop=True, inplace=True)
amenities.reset_index(drop=True, inplace=True)
amenities_data = pd.concat([hotels, amenities, stations], ignore_index=True).drop_duplicates()
amenities_data['Postal Code'] = amenities_data['Postal Code'].str.strip()
amenities_data['LSOA Code'] = amenities_data['Postal Code'].map(lsoa_postcode_mapping)

# Count amenities per LSOA Code and merge with LSOA data
amenities_count = amenities_data.groupby('LSOA Code').size().reset_index(name='Amenities')
lsoa_data = pd.merge(lsoa_data, amenities_count, on='LSOA Code', how='left').fillna({'Amenities': 0})



###**1.5 Crime Rate**

Calculating the total crime for the period from June 2023 to June 2024.

In [ ]:
import pandas as pd

# Load Crime Data
crime_data = pd.read_csv('MPS LSOA Level Crime (most recent 24 months).csv')

# Define the columns to include for the sum (excluding 'LSOA Code')
crime_columns_to_include = [f'2023{month:02}' for month in range(6, 13)] + [f'2024{month:02}' for month in range(1, 7)]

# Convert the selected columns to numeric, replacing non-numeric values with 0
crime_data[crime_columns_to_include] = crime_data[crime_columns_to_include].apply(pd.to_numeric, errors='coerce').fillna(0)

# Calculate the total crime per LSOA by summing across the selected months
crime_data['Total Crime'] = crime_data[crime_columns_to_include].sum(axis=1)

# Aggregate by LSOA Code to ensure one row per LSOA Code
crime_data = crime_data.groupby('LSOA Code', as_index=False)['Total Crime'].sum()

# Merge Crime Data and Calculate Crime Rate per 1000 Residents
# Ensure 'Population 2015' exists in lsoa_data
merged_data = pd.merge(lsoa_data, crime_data[['LSOA Code', 'Total Crime']], on='LSOA Code', how='left')

# If your population data is from 2015, ensure the column name matches exactly
merged_data['Crime Rate per 1000'] = (merged_data['Total Crime'] / merged_data['Population 2015']) * 1000
merged_data['Crime Rate per 1000'] = merged_data['Crime Rate per 1000'].round(2)

# Save the Updated Data
merged_data.to_csv('LSOA Statistics.csv', index=False)
print("The updated LSOA data has been saved to 'LSOA Statistics.csv'.")


###**1.6 Property Data**

In [ ]:
# Load the dataset
file_path = 'Property Data.csv'
property_data = pd.read_csv(file_path)

# Display the first few rows of the dataset to understand its structure
property_data.head()

In [ ]:
# Function to extract postcode from an address
def extract_postcode(address):
    postcode_regex = r'[A-Z]{1,2}\d{1,2}[A-Z]?\s?\d[A-Z]{2}'
    match = re.search(postcode_regex, address)
    if match:
        return match.group(0)
    return None

# Apply the function to extract postcodes
property_data['Extracted Postcode'] = property_data['Address'].apply(extract_postcode)

# Fill missing Postcode values with the extracted postcodes
property_data['Postcode'].fillna(property_data['Extracted Postcode'], inplace=True)

# Drop the 'Extracted Postcode' column as it's no longer needed
property_data.drop(columns=['Extracted Postcode'], inplace=True)

# Function to clean the Size column
def clean_size(size):
    if isinstance(size, str):
        # Remove commas and 'Sq Ft'
        size_cleaned = size.replace(',', '').replace('Sq Ft', '').strip()

        # Check if there's a range (e.g., '1,000-2,000')
        if '-' in size_cleaned:
            # Split the range and take the smallest value
            size_cleaned = size_cleaned.split('-')[0]

        # Convert to float first to handle any decimal points, then to int
        try:
            return int(float(size_cleaned))
        except ValueError:
            return None
    else:
        # If it's not a string, it could be NaN, return as is
        return None

# Apply the function to clean the Size column
property_data['Size'] = property_data['Size'].apply(clean_size)

# Rename the Rent column
property_data.rename(columns={'Rent (per sq foot)': 'Rent (per sq.ft)'}, inplace=True)

# Drop the Type and Address columns
property_data.drop(columns=['Type', 'Address', 'Size'], inplace=True)


# Format the 'Rent (per sq foot)' column to have 2 decimal places
property_data['Rent (per sq.ft)'] = property_data['Rent (per sq.ft)'].round(2)


# Remove duplicates
property_data = property_data.drop_duplicates()

# Remove rows with any missing values
property_data = property_data.dropna()

property_data.to_csv('Rent Prices.csv', index=False)
# Get the shape of the dataset
print('\nDataset shape:\n',property_data.shape)


##**2. Exploratory Data Analysis (EDA)**

###**Dataset Reordering and renaming**

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import re

In [ ]:
# Load the main LSOA dataset
data = pd.read_csv('LSOA Statistics.csv')

# Specify the desired column order
desired_order = [

    'LSOA Code',
    'LSOA Name',
    'Postcode',
    'Population 2015',
    'Average Age 2015',
    'Average Income',
    'Employment Rate 2011',
    'PT Accessibility Levels 2014',
    'Median House Price 2023',
    'Index of Multiple Deprivation',
    'Distance to Station',
    'Crime Rate per 1000',
    'Amenities',
    'Competitors',
    'Cafe_Score',
    'Reviews'
]

# Reorder the DataFrame columns
data = data[desired_order]


###**2.1 Descriptive Statistics**

In [ ]:
# Get the shape of the dataset
print('Dataset shape:\n', data.shape)

The final dataset has 4835 rows and 16 columns.



In [ ]:
print(data.info())

Here, the Median House Price 2023 column is identified as having an object data type, which suggests the presence of non-numeric entries. This raises concerns about data quality and necessitates further investigation. To address this, we first examine the non-numeric values in the column.

LSOA Code, LSOA Name and Postcode columns are the metadata columns that would be needed later for visualisations.


In [ ]:
# Find non-numeric values in the 'Median House Price 2023' column
non_numeric_values = data[~data['Median House Price 2023'].str.replace(',', '').str.isnumeric()]
print("Non-numeric values in 'Median House Price 2023':\n", non_numeric_values[['LSOA Code', 'Median House Price 2023']])

Upon inspection, we found that the values ':' are listed, which is unusual to find in a numeric column. To clean the data, all entries were first converted to strings, and non-numeric values were identified and replaced with NaN to be imputed in the data cleaning stage. This step is essential at this point to ensure that the column is correctly formatted for subsequent analysis and modeling.

In [ ]:
# Convert non-numeric entries to NaN
data['Median House Price 2023'] = pd.to_numeric(data['Median House Price 2023'].str.replace(',', ''), errors='coerce')

In [ ]:
# Summary statistics for numerical features
numerical_columns = ['Population 2015', 'Average Age 2015', 'Average Income', 'Employment Rate 2011',
                     'PT Accessibility Levels 2014', 'Median House Price 2023', 'Index of Multiple Deprivation',
                     'Distance to Station', 'Crime Rate per 1000', 'Amenities', 'Competitors', 'Cafe_Score', 'Reviews']
data[numerical_columns].describe()

In [ ]:
data[numerical_columns]

###**2.2 Visualizing Distributions**

In [ ]:
# Visualize the numerical features
# Set up subplots
num_plots = len(numerical_columns)
fig, axes = plt.subplots(nrows=num_plots//3 + 1, ncols=3, figsize=(18, 12))

# Plot histograms for each feature
axes = axes.flatten()  # Flatten the axes array to easily iterate over it

for i, feature in enumerate(numerical_columns):
    axes[i].hist(data[feature], bins=20, color='skyblue', edgecolor='black')
    axes[i].set_title(f'Histogram of {feature}')
    axes[i].set_xlabel(feature)
    axes[i].set_ylabel('Frequency')

# Hide any empty subplots (in case of a non-square grid)
for j in range(i + 1, len(axes)):
    fig.delaxes(axes[j])

plt.tight_layout()
plt.show()

The visualizations provide a comprehensive overview of various socio-economic metrics across LSOA areas in London. The histograms reveal the frequency distribution of metrics like population, income, and house prices, showing how values are concentrated across different ranges. For example, many metrics are skewed towards the lower end (such as crime rates and house prices), while others, like average age and employment rates, exhibit a more normal distribution.

In [ ]:
# Set up subplots
num_plots = len(numerical_columns)
fig, axes = plt.subplots(nrows=num_plots//3 + 1, ncols=3, figsize=(18, 12))

# Flatten the axes array to easily iterate over it
axes = axes.flatten()

for i, feature in enumerate(numerical_columns):
    # Drop any NaN values from the column before plotting
    clean_data = data[feature].dropna()
    # Ensure the data is numeric
    if clean_data.dtype.kind in 'if':  # 'i' for integer, 'f' for float
        axes[i].boxplot(clean_data, patch_artist=True, boxprops=dict(facecolor='skyblue'))
        axes[i].set_title(f'Boxplot of {feature}')
        axes[i].set_xlabel(feature)
        axes[i].set_ylabel('Value')

# Hide any empty subplots (in case of a non-square grid)
for j in range(i + 1, len(axes)):
    fig.delaxes(axes[j])

plt.tight_layout()
plt.show()


The boxplots highlight these metrics' distribution, variability, and presence of outliers. They emphasize disparities across different areas, such as significantly higher or lower values in certain regions for metrics like income and house prices. Together, these visualizations provide a clear and detailed picture of the socio-economic landscape in London's LSOAs, helping to identify patterns and anomalies across the city.

###**2.3 Checking for duplicates and missing values**

In [ ]:
print(f'There are {data.duplicated().sum()} duplicates rows.')

In [ ]:
print(data.isnull().sum())

From the above output it was observed that most of the features do not contain missing values. However, the 'Median House Price 2023' and 'Crime Rate per 1000' features have 386 and 182 missing values respectively. This is an important observation as missing data can impact the performance of models and the accuracy of analysis. Given that both are significant features related to the context of understanding the success of a café in a particular area, careful handling of these missing values will be necessary in the preprocessing stage. Particularly, these missing values should be imputed with the mean value for the column.

##**3. Data Cleaning**

###**3.1 Managing Missing Values**

Filling in Missing Data (Imputation method)

In [ ]:
print('Number of missing values for Median House Prices: ', data['Median House Price 2023'].isna().sum())
print('Number of missing values for Crime Rate: ', data['Crime Rate per 1000'].isna().sum())

In [ ]:
from sklearn.impute import SimpleImputer
import numpy as np

# Imputer for the 'Median House Prices 2023' column
imptr_house_price = SimpleImputer(missing_values=np.nan, strategy='mean')

# Fit the imputer and transform the data
data['Median House Price 2023'] = imptr_house_price.fit_transform(data[['Median House Price 2023']])
data['Crime Rate per 1000'] = imptr_house_price.fit_transform(data[['Crime Rate per 1000']])

# Check for missing values after imputation
print('Missing Values:\n', data.isnull().sum())

##**4. Data Preprocessing**

###**4.1 Feature Selection**

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

# Select only the relevant numerical columns (excluding the first 3)
numerical_columns = data.columns[3:]

# Calculate the correlation matrix
corr_matrix = data[numerical_columns].corr()

# Plotting the heatmap
plt.figure(figsize=(12, 8))
sns.heatmap(corr_matrix, annot=True, cmap='coolwarm')
plt.title('Correlation Matrix (Excluding First 3 Columns)')
plt.show()



After analyzing the correlation matrix, it was observed that the `Reviews` feature has a high correlation (0.81) with the `Competitors` feature. This high correlation suggests that `Reviews` and `Competitors` capture similar information. To reduce multicollinearity and avoid redundancy in the model, the `Reviews` column ahould be dropped. Retaining `Competitors` is preferred as it provides more direct insights into the competitive landscape, which is a crucial factor for the success of a new café in the area.


In [ ]:
# Dropping the 'Cafe Reviews' column
data = data.drop(columns=['Reviews'])

# Confirm that the column has been dropped
print(data.columns)


###**4.2 Feature Normalization**

Since many features are skewed and have outliers, normalization will help bring them to a common scale, which is essential before applying clustering algorithms.


In [ ]:
from sklearn.preprocessing import MinMaxScaler
from sklearn.cluster import KMeans
import matplotlib.pyplot as plt
import seaborn as sns

# Step 1: Normalize the data (selecting only numeric features)
numeric_features = [
    'Population 2015', 'Average Age 2015', 'Average Income',
    'Employment Rate 2011', 'PT Accessibility Levels 2014',
    'Median House Price 2023', 'Index of Multiple Deprivation',
    'Distance to Station', 'Crime Rate per 1000', 'Amenities',
    'Competitors', 'Cafe_Score'
]
scaler = MinMaxScaler()
normalized_features = scaler.fit_transform(data[numeric_features])
